<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula18a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False, parser="auto")
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

In [2]:
from sklearn.linear_model import RidgeClassifier
model = RidgeClassifier()

def evaluate(model, data):
  X_train, y_train, X_test, y_test = data
  model.fit(X_train, y_train)
  return model.score(X_test, y_test)

data_original = (X_train, y_train, X_test, y_test)

evaluate(model, data_original)

0.8603

Redução de dimensionalidade por extração de características a partir do domínio das características

In [3]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

data_pca = (X_train_pca, y_train, X_test_pca, y_test)

evaluate(model, data_pca)

0.8609

In [7]:
d = X_train_pca.shape[1]
d

154

# Redução de dimensionalidade por seleção de características

Pode melhorar em situações em que exista características redundantes ou irrelevantes.

Se não feita com cuidado resulta em overfiting e resultados enviesados.

# Tipos de seleção de características

*   Filter
*   Wrapper
*   Embedded

Todos pode ser unimodal ou multimodal.



In [28]:
import numpy as np

def varfs(data, d):
  X_train, y_train, X_test, y_test = data
  variances = X_train.var(axis=0)
  feat_idx = np.argsort(variances)[::-1]
  X_train_varfs = X_train[:, feat_idx[:d]]
  X_test_varfs = X_test[:, feat_idx[:d]]
  data_varfs = (X_train_varfs, y_train, X_test_varfs, y_test)
  return data_varfs

data_varfs = varfs(data_original, d)
evaluate(model, data_varfs)

0.7955

In [48]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0)
selector.fit(X_train)
X_train_varfs = selector.transform(X_train)
X_test_varfs = selector.transform(X_test)
data_varfs = (X_train_varfs, y_train, X_test_varfs, y_test)

print(X_train_varfs.shape)
evaluate(model, data_varfs)

(60000, 695)


0.8604

In [49]:
from sklearn.feature_selection import SelectKBest, f_classif
X_train, y_train, X_test, y_test = data_varfs
selector = SelectKBest(f_classif, k=d)
selector.fit(X_train, y_train)
X_train_kbest = selector.transform(X_train)
X_test_kbest = selector.transform(X_test)
data_kbest = (X_train_kbest, y_train, X_test_kbest, y_test)

evaluate(model, data_kbest)

0.8152

In [ ]:
from sklearn.model_selection import train_test_split

def sequential_forward_selection(model, data, d):
  X_train, y_train, X_test, y_test = data
  X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)
  best_feats = []
  remaining_feats = list(range(X_tr.shape[1]))
  for _ in range(d):
    best_score = -np.inf
    best_feat = -1
    for i in remaining_feats:
      print(10*"\b", f"{i}/{len(remaining_feats)}", end="")
      best_feats.append(i)
      model.fit(X_tr[:, best_feats], y_tr)
      score = model.score(X_val[:, best_feats], y_val)
      if score > best_score:
        best_score = score
        best_feat = i
      best_feats = best_feats[:-1]
    print(best_score, best_feat)
    best_feats.append(best_feat)
    remaining_feats.remove(best_feat)
  data_fwsel = (X_train[:, best_feats], y_train, X_test[:, best_feats], y_test)
  return data_fwsel

data_fwsel = sequential_forward_selection(model, data_original, d)
evaluate(model, data_fwsel)

 0/784 1/784 2/784 3/784 4/784 5/784 6/784 7/784 8/784 9/784 10/784 11/784 12/784 13/784 14/784 15/784 16/784 17/784 18/784 19/784 20/784 21/784 22/784 23/784 24/784 25/784 26/784 27/784 28/784 29/784 30/784 31/784 32/784 33/784 34/784 35/784 36/784 37/784 38/784 39/784 40/784 41/784 42/784 43/784 44/784 45/784 46/784 47/784 48/784 49/784 50/784 51/784 52/784 53/784 54/784 55/784 56/784 57/784 58/784